# Pipeline Gold — Pré-processamento ML-Ready

**Objetivo:** unir os 3 datasets Silver, aplicar o pipeline de transformações (encoding,
scaling, missing values, outliers) respeitando o padrão fit/transform, e salvar o
dataset ML-ready em Parquet.

**Branch:** `feature/gold-layer-pre-processing`

---

| Etapa | Descrição |
|-------|-----------|
| 1 | Setup — imports, paths, pastas |
| 2 | Leitura Silver + sanity check |
| 3 | Join controlado (incidents ← financial ← market) |
| 4 | Anti-leakage final |
| 5 | Definição de grupos de colunas |
| 6 | Split estratificado 80/20 |
| 7 | Construção do ColumnTransformer |
| 8 | fit/transform |
| 9 | Reconstrução do DataFrame final |
| 10 | Salvar dataset_ml_ready.parquet + preprocessor.joblib |
| 11 | Validação final |
| 12 | Gerar docs/gold_transformations.md |

In [30]:
# Etapa 1 — Setup
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, RobustScaler,
    OneHotEncoder, FunctionTransformer, TargetEncoder,
)
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
import joblib

# Paths
PROJECT_ROOT = Path.cwd().parent
SILVER_PATH  = PROJECT_ROOT / "data" / "silver"
GOLD_PATH    = PROJECT_ROOT / "data" / "gold"
MODELS_PATH  = PROJECT_ROOT / "models"
DOCS_PATH    = PROJECT_ROOT / "docs"

# Garantir pastas
GOLD_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

print("Paths configurados:")
print(" Silver:", SILVER_PATH)
print(" Gold:  ", GOLD_PATH)
print(" Models:", MODELS_PATH)

Paths configurados:
 Silver: c:\Users\dti-\Pessoal\CC5\ciencia-de-dados\cybersecurity-breach-data-project\data\silver
 Gold:   c:\Users\dti-\Pessoal\CC5\ciencia-de-dados\cybersecurity-breach-data-project\data\gold
 Models: c:\Users\dti-\Pessoal\CC5\ciencia-de-dados\cybersecurity-breach-data-project\models


---
## Etapa 2 — Leitura Silver + Sanity Check

In [31]:
# Etapa 2 — Leitura Silver + sanity check
df_inc = pd.read_parquet(SILVER_PATH / "incidents_master_silver.parquet")
df_fin = pd.read_parquet(SILVER_PATH / "financial_impact_silver.parquet")
df_mkt = pd.read_parquet(SILVER_PATH / "market_impact_silver.parquet")

# --- Shapes ---
print("=== Shapes ===")
print(f"incidents_master : {df_inc.shape}")
print(f"financial_impact : {df_fin.shape}")
print(f"market_impact    : {df_mkt.shape}")

# --- Unicidade da chave de join ---
print("\n=== Duplicatas em incident_id ===")
for name, df in [("incidents", df_inc), ("financial", df_fin), ("market", df_mkt)]:
    n = df["incident_id"].duplicated().sum()
    status = "✅ OK" if n == 0 else f"⚠️  {n} duplicatas"
    print(f"  {name:12s}: {status}")

# --- Nulos relevantes ---
print("\n=== Nulos por dataset ===")
for name, df in [("incidents", df_inc), ("financial", df_fin), ("market", df_mkt)]:
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    if len(nulos) == 0:
        print(f"  {name}: nenhum nulo")
    else:
        print(f"  {name}:")
        for col, n in nulos.items():
            pct = n / len(df) * 100
            print(f"    {col:35s}: {n:4d} ({pct:5.1f}%)")

# --- Distribuição do label ---
print("\n=== Distribuição do label (incidents) ===")
dist = df_inc["label_severe_incident"].value_counts(normalize=True).mul(100).round(1)
for k, v in dist.items():
    print(f"  Classe {k}: {v}%")

=== Shapes ===
incidents_master : (849, 28)
financial_impact : (778, 15)
market_impact    : (358, 28)

=== Duplicatas em incident_id ===
  incidents   : ✅ OK
  financial   : ✅ OK
  market      : ✅ OK

=== Nulos por dataset ===
  incidents:
    stock_ticker                       :  438 ( 51.6%)
  financial:
    ransom_demanded_usd                :  572 ( 73.5%)
    ransom_paid_usd                    :  692 ( 88.9%)
    regulatory_fine_usd                :  646 ( 83.0%)
  market: nenhum nulo

=== Distribuição do label (incidents) ===
  Classe 1: 86.1%
  Classe 0: 13.9%


---
## Etapa 3 — Join Controlado

Estratégia:
- `incidents LEFT JOIN financial` — 91,5% dos incidents têm match (~778/849)
- `incidents LEFT JOIN market`   — 42,1% dos incidents têm match (~357/849)

Incidents é o **fato central**; financial e market são dimensões opcionais.
Nulos gerados pelo LEFT join em linhas sem match são estruturais e serão tratados
no pipeline com `SimpleImputer + add_indicator`.

In [32]:
# Etapa 3 — Join controlado
df = (
    df_inc
    .merge(df_fin, on="incident_id", how="left", suffixes=("", "_fin"))
    .merge(df_mkt, on="incident_id", how="left", suffixes=("", "_mkt"))
)

print(f"Shape após join: {df.shape}")
print(f"Esperado:        ({len(df_inc)}, {df_inc.shape[1] + df_fin.shape[1] + df_mkt.shape[1] - 2})")

# Taxa de match por dimensão
n_fin = df["total_loss_usd"].notna().sum()
n_mkt = df["market_cap_at_disclosure"].notna().sum()
n_total = len(df)

print("\n=== Taxa de match pós-join ===")
print(f"  incidents (base)  : {n_total:4d} linhas (100%)")
print(f"  com financial     : {n_fin:4d} linhas ({n_fin/n_total*100:.1f}%)")
print(f"  sem financial     : {n_total-n_fin:4d} linhas ({(n_total-n_fin)/n_total*100:.1f}%)")
print(f"  com market        : {n_mkt:4d} linhas ({n_mkt/n_total*100:.1f}%)")
print(f"  sem market        : {n_total-n_mkt:4d} linhas ({(n_total-n_mkt)/n_total*100:.1f}%)")

# Verificar se label foi preservado
assert df["label_severe_incident"].notna().all(), "ERRO: nulos no label após join!"
print(f"\n✅ label_severe_incident preservado em todas as {n_total} linhas")

# Confirmar que não há linhas duplicadas pós-join (garantia de chave 1:1)
n_dup = df.duplicated(subset=["incident_id"]).sum()
print(f"✅ Duplicatas pós-join: {n_dup} (esperado: 0)")

Shape após join: (849, 69)
Esperado:        (849, 69)

=== Taxa de match pós-join ===
  incidents (base)  :  849 linhas (100%)
  com financial     :  777 linhas (91.5%)
  sem financial     :   72 linhas (8.5%)
  com market        :  357 linhas (42.0%)
  sem market        :  492 linhas (58.0%)

✅ label_severe_incident preservado em todas as 849 linhas
✅ Duplicatas pós-join: 0 (esperado: 0)


---
## Etapa 4 — Revisão Anti-Leakage Final

**Estratégia em duas etapas:**

1. **Antes de dropar**: salvar as colunas pós-evento em `market_retroactive.parquet` para uso futuro (análise retroativa, não disponível em produção).
2. **Dropar** identificadores, datas cruas e colunas pós-evento do dataframe principal.

| Grupo | Colunas | Motivo |
|-------|---------|--------|
| Identificadores | `incident_id`, `stock_ticker`, `stock_ticker_mkt` | Não são features |
| Datas cruas | `incident_date`, `incident_date_estimated`, `discovery_date` | Já derivadas em `days_to_*` |
| Granularidade temporal inútil | `incident_month`, `incident_day` | DTs não se beneficiam de dia/mês isolados |
| Preços pós-evento | `price_1d_after`, `price_7d_after`, `price_30d_after` | Leakage: ocorrem após o incidente |
| Retornos anormais | `abnormal_return_{1d,7d,30d}` | Idem |
| CAR | `car_{neg1_to_pos1, 0_to_7, 0_to_30, 0_to_90}` | Idem |
| Volatilidade pós | `post_incident_volatility_30d` | Idem |
| Recuperação de preço | `days_to_price_recovery` | Idem |

In [33]:
# Etapa 4 — Anti-leakage final

# -------------------------------------------------------------------
# 4a. Colunas pós-evento → salvar separadamente ANTES de dropar
# -------------------------------------------------------------------
COLS_RETROACTIVE = [
    "incident_id",
    "price_1d_after", "price_7d_after", "price_30d_after",
    "abnormal_return_1d", "abnormal_return_7d", "abnormal_return_30d",
    "car_neg1_to_pos1", "car_0_to_7", "car_0_to_30", "car_0_to_90",
    "post_incident_volatility_30d",
    "days_to_price_recovery",
]

# Apenas linhas com dados de mercado (as demais seriam NaN em todas as cols pós-evento)
retro_subset = [c for c in COLS_RETROACTIVE if c != "incident_id"]
df_retro = df.loc[df[retro_subset].notna().any(axis=1), COLS_RETROACTIVE].copy()
df_retro.to_parquet(GOLD_PATH / "market_retroactive.parquet", index=False)
print(f"market_retroactive.parquet salvo: {df_retro.shape} linhas × colunas")

# -------------------------------------------------------------------
# 4b. Definir e dropar todas as colunas de leakage
# -------------------------------------------------------------------
COLS_LEAKAGE = [
    # Identificadores
    "incident_id", "stock_ticker", "stock_ticker_mkt",
    # Datas cruas (features temporais já derivadas em days_to_*)
    # NB: incident_date_estimated (bool) é mantida — indica qualidade dos dados
    "incident_date", "discovery_date",
    # Granularidade temporal sem valor para DT
    "incident_month", "incident_day",
    # Preços pós-evento
    "price_1d_after", "price_7d_after", "price_30d_after",
    # Retornos anormais pós-evento
    "abnormal_return_1d", "abnormal_return_7d", "abnormal_return_30d",
    # Cumulative Abnormal Return pós-evento
    "car_neg1_to_pos1", "car_0_to_7", "car_0_to_30", "car_0_to_90",
    # Volatilidade e recuperação pós-evento
    "post_incident_volatility_30d",
    "days_to_price_recovery",
]

# Dropar apenas colunas que existam (proteção contra mudanças de schema)
cols_to_drop = [c for c in COLS_LEAKAGE if c in df.columns]
cols_missing  = [c for c in COLS_LEAKAGE if c not in df.columns]

df = df.drop(columns=cols_to_drop)

print(f"\nShape pós anti-leakage: {df.shape}")
print(f"Colunas removidas ({len(cols_to_drop)}): {cols_to_drop}")
if cols_missing:
    print(f"⚠️  Colunas declaradas mas ausentes no df: {cols_missing}")
else:
    print("✅ Todas as colunas de leakage declaradas estavam presentes")

market_retroactive.parquet salvo: (357, 13) linhas × colunas

Shape pós anti-leakage: (849, 50)
Colunas removidas (19): ['incident_id', 'stock_ticker', 'stock_ticker_mkt', 'incident_date', 'discovery_date', 'incident_month', 'incident_day', 'price_1d_after', 'price_7d_after', 'price_30d_after', 'abnormal_return_1d', 'abnormal_return_7d', 'abnormal_return_30d', 'car_neg1_to_pos1', 'car_0_to_7', 'car_0_to_30', 'car_0_to_90', 'post_incident_volatility_30d', 'days_to_price_recovery']
✅ Todas as colunas de leakage declaradas estavam presentes


---
## Etapa 5 — Definição dos Grupos de Colunas

O `ColumnTransformer` exige que cada coluna seja atribuída a **exatamente um** grupo.
A célula abaixo define os grupos e verifica cobertura total (sem lacunas, sem sobreposições).

| Grupo | Transformer | Critério |
|-------|-------------|----------|
| `num_normal` | `SimpleImputer(median) → IQRClipper → StandardScaler` | Numéricas sem escala monetária |
| `num_monetary` | `SimpleImputer(median) → log1p → RobustScaler` | Valores em USD ou de escala monetária |
| `cat_lowcard` | `SimpleImputer(constant) → OneHotEncoder` | Categóricas com ≤ 20 categorias |
| `cat_highcard` | `SimpleImputer(constant) → TargetEncoder` | Categóricas com > 20 categorias |
| `binary_cols` | `passthrough` | Flags 0/1 ou bool já codificadas |

In [34]:
# Etapa 5 — Definição dos grupos de colunas
from collections import Counter

TARGET = "label_severe_incident"

# ------------------------------------------------------------------
# Numéricas sem escala monetária → SimpleImputer(median) + IQRClipper + StandardScaler
# ------------------------------------------------------------------
num_normal = [
    # Incidentes — temporalidade e dimensão operacional
    "incident_year",
    "days_to_discovery",
    "days_to_disclosure",
    "employee_count",
    "downtime_hours",
    "data_compromised_records",
    # Mercado — volume e estatísticas de retorno (não USD diretos)
    "volume_avg_30d_baseline",
    "volume_disclosure_day",
    "sector_return_same_period",
    "t_statistic_1d",
    "p_value_1d",
    "t_statistic_30d",
    "p_value_30d",
    "volume_ratio_disclosure",
    "pre_incident_volatility_30d",
]

# ------------------------------------------------------------------
# Numéricas monetárias (USD / preços) → SimpleImputer(median) + log1p + RobustScaler
# ------------------------------------------------------------------
num_monetary = [
    # Incidentes
    "company_revenue_usd",
    # Financeiras
    "direct_loss_usd",
    "ransom_demanded_usd",
    "ransom_paid_usd",
    "recovery_cost_usd",
    "legal_fees_usd",
    "regulatory_fine_usd",
    "insurance_payout_usd",
    "total_loss_usd",
    "total_loss_lower_bound",
    "total_loss_upper_bound",
    "inflation_adjusted_usd",
    # Mercado — preços e capitalização
    "price_7d_before",
    "price_disclosure_day",
    "market_cap_at_disclosure",
]

# ------------------------------------------------------------------
# Categóricas baixa cardinalidade (≤ 20 cats) → OneHotEncoder
# ------------------------------------------------------------------
cat_lowcard = [
    "attack_vector_primary",
    "attribution_confidence",
    "data_type",
    "data_source_type",
    "sector_index",          # 10 valores únicos (S&P setores)
]

# ------------------------------------------------------------------
# Categóricas alta cardinalidade (> 20 cats) → TargetEncoder
# ------------------------------------------------------------------
cat_highcard = [
    "industry_primary",      # ~20+ setores industriais
    "country_hq",            # 50+ países
    "attributed_group",      # grupos de ameaça variados
]

# ------------------------------------------------------------------
# Binárias (já codificadas 0/1 ou bool) → passthrough
# ------------------------------------------------------------------
binary_cols = [
    # Incidentes
    "is_public_company",
    "incident_date_estimated",   # bool
    "has_secondary_vector",
    "data_loss_unknown",
    "downtime_unknown",
    "has_data_loss",
    "has_downtime",
    # Financeiras
    "is_ransomware",
    "has_regulatory_fine",
    "insurance_unknown",
    # Mercado
    "earnings_announcement_within_7d",
]

# ------------------------------------------------------------------
# Verificação de cobertura completa
# ------------------------------------------------------------------
all_feature_cols = sorted(c for c in df.columns if c != TARGET)
all_assigned     = num_normal + num_monetary + cat_lowcard + cat_highcard + binary_cols

# Duplicatas entre grupos
dupes = [c for c, n in Counter(all_assigned).items() if n > 1]

# Colunas do df não atribuídas a nenhum grupo
missing_from_groups = [c for c in all_feature_cols if c not in set(all_assigned)]

# Colunas declaradas mas ausentes no df (ex: typo)
ghost_cols = [c for c in all_assigned if c not in df.columns]

print(f"Total de features no df       : {len(all_feature_cols)}")
print(f"Total de features em grupos   : {len(set(all_assigned))}")
print(f"Duplicatas entre grupos       : {dupes if dupes else 'nenhuma ✅'}")
print(f"Sem grupo (serão dropadas)    : {missing_from_groups if missing_from_groups else 'nenhuma ✅'}")
print(f"Declaradas mas ausentes no df : {ghost_cols if ghost_cols else 'nenhuma ✅'}")

print("\n--- Resumo dos grupos ---")
for name, lst in [
    ("num_normal  ", num_normal),
    ("num_monetary", num_monetary),
    ("cat_lowcard ", cat_lowcard),
    ("cat_highcard", cat_highcard),
    ("binary_cols ", binary_cols),
]:
    print(f"  {name}: {len(lst):2d} colunas")

Total de features no df       : 49
Total de features em grupos   : 49
Duplicatas entre grupos       : nenhuma ✅
Sem grupo (serão dropadas)    : nenhuma ✅
Declaradas mas ausentes no df : nenhuma ✅

--- Resumo dos grupos ---
  num_normal  : 15 colunas
  num_monetary: 15 colunas
  cat_lowcard :  5 colunas
  cat_highcard:  3 colunas
  binary_cols : 11 colunas


---
## Etapa 6 — Split Estratificado 80/20

**Justificativa do stratify:** label desbalanceado (86% / 14%) — sem estratificação,
split aleatório pode concentrar a classe minoritária em apenas um dos conjuntos.

O split ocorre **antes** de qualquer `fit`, garantindo que medianas, top-categorias
e limites IQR sejam calculados exclusivamente sobre o treino.

In [35]:
# Etapa 6 — Split estratificado 80/20
from sklearn.model_selection import train_test_split

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print(f"X_train : {X_train.shape}  |  X_test : {X_test.shape}")
print(f"y_train : {y_train.shape}  |  y_test : {y_test.shape}")

# Verificar que a proporção do label foi preservada
print("\n=== Distribuição do label por split ===")
for split_name, y_split in [("treino", y_train), ("teste ", y_test), ("total ", y)]:
    dist = y_split.value_counts(normalize=True).mul(100).round(1)
    print(f"  {split_name} — classe 0: {dist.get(0, 0.0)}%  |  classe 1: {dist.get(1, 0.0)}%  (n={len(y_split)})")

X_train : (679, 49)  |  X_test : (170, 49)
y_train : (679,)  |  y_test : (170,)

=== Distribuição do label por split ===
  treino — classe 0: 13.8%  |  classe 1: 86.2%  (n=679)
  teste  — classe 0: 14.1%  |  classe 1: 85.9%  (n=170)
  total  — classe 0: 13.9%  |  classe 1: 86.1%  (n=849)


---
## Etapa 7 — Construção do ColumnTransformer

### Decisões por grupo

| Grupo | Passos | Notas |
|-------|--------|-------|
| `num_normal` | `IQRClipper(k=1.5)` → `SimpleImputer(median, add_indicator)` → `StandardScaler` | IQR antes da imputação (usa `nanpercentile`); indicadores sinalizam NaN estrutural |
| `num_monetary` | `SimpleImputer(median, add_indicator)` → `log1p` → `RobustScaler` | `log1p` comprime caudas longas (vide EDA G5); RobustScaler robusto a outliers residuais |
| `cat_lowcard` | `SimpleImputer(constant="unknown")` → `OneHotEncoder(min_frequency=10)` | Categorias raras (< 10 ocorrências no treino) viram `infrequent_sklearn` |
| `cat_highcard` | `SimpleImputer(constant="unknown")` → `TargetEncoder` | Evita explosão dimensional; encoding supervisionado, fit só no treino |
| `binary_cols` | `SimpleImputer(constant=0)` | NaN estruturais (sem dados de mercado/financeiro) → 0 semanticamente correto |

`remainder="drop"` descarta qualquer coluna não listada explicitamente.

In [36]:
# Etapa 7 — Construção do ColumnTransformer

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, RobustScaler,
    OneHotEncoder, FunctionTransformer, TargetEncoder,
)
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin


# ------------------------------------------------------------------
# 7a. IQRClipper — clipper fit/transform-aware para outliers
# ------------------------------------------------------------------
class IQRClipper(BaseEstimator, TransformerMixin):
    """Clipa valores fora do intervalo [Q1 - k*IQR, Q3 + k*IQR].
    Ignora NaN no fit (nanpercentile). Não modifica NaN no transform.
    """
    def __init__(self, k=1.5):
        self.k = k

    def fit(self, X, y=None):
        q1 = np.nanpercentile(X, 25, axis=0)
        q3 = np.nanpercentile(X, 75, axis=0)
        iqr = q3 - q1
        self.lower_ = q1 - self.k * iqr
        self.upper_ = q3 + self.k * iqr
        return self

    def transform(self, X):
        # np.clip preserva NaN; IQRClipper não imputa
        return np.clip(X, self.lower_, self.upper_)

    def get_feature_names_out(self, input_features=None):
        return input_features if input_features is not None else np.arange(self.lower_.shape[0]).astype(str)


# ------------------------------------------------------------------
# 7b. Sub-pipelines
# ------------------------------------------------------------------

# Numéricas sem escala monetária
# IQRClipper antes da imputação (nanpercentile ignora NaN)
num_normal_pipe = Pipeline([
    ("clip",   IQRClipper(k=1.5)),
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale",  StandardScaler()),
])

# Numéricas monetárias (USD) — log1p comprime cauda longa
num_money_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("log",    FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scale",  RobustScaler()),
])

# Categóricas baixa cardinalidade — OneHot
cat_low_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("ohe",    OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        sparse_output=False,
    )),
])

# Categóricas alta cardinalidade — TargetEncoder supervisionado
cat_high_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("te",     TargetEncoder(target_type="binary")),
])

# Binárias (flags 0/1 ou bool) — NaN estrutural → 0
binary_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value=0)),
])


# ------------------------------------------------------------------
# 7c. ColumnTransformer final
# ------------------------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num_norm",  num_normal_pipe,  num_normal),
        ("num_money", num_money_pipe,   num_monetary),
        ("cat_low",   cat_low_pipe,     cat_lowcard),
        ("cat_high",  cat_high_pipe,    cat_highcard),
        ("binary",    binary_pipe,      binary_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

print("ColumnTransformer construído com", len(preprocessor.transformers), "transformers:")
for name, pipe, cols in preprocessor.transformers:
    steps = " → ".join(s for s, _ in pipe.steps)
    print(f"  [{name:10s}] {len(cols):2d} colunas  |  {steps}")

ColumnTransformer construído com 5 transformers:
  [num_norm  ] 15 colunas  |  clip → impute → scale
  [num_money ] 15 colunas  |  impute → log → scale
  [cat_low   ]  5 colunas  |  impute → ohe
  [cat_high  ]  3 colunas  |  impute → te
  [binary    ] 11 colunas  |  impute


---
## Etapa 8 — fit no treino + transform em treino e teste

**Regra crítica:** `fit` e `fit_transform` são chamados **apenas** sobre `X_train`.
`X_test` passa apenas pelo `transform` — garantia de que nenhuma estatística
(mediana, limites IQR, encodings) vaza do conjunto de teste para o treino.

O `TargetEncoder` recebe `y_train` no fit pois é um encoder supervisionado.

In [37]:
# Etapa 8 — fit no treino + transform em treino e teste

# fit: usa y_train porque TargetEncoder é supervisionado
preprocessor.fit(X_train, y_train)

# transform: NUNCA passa y no transform (evita leakage)
X_train_t = preprocessor.transform(X_train).astype(np.float64)
X_test_t  = preprocessor.transform(X_test).astype(np.float64)

print(f"X_train transformado: {X_train_t.shape}")
print(f"X_test  transformado: {X_test_t.shape}")

# Verificar ausência de NaN/Inf no output
nan_train = np.isnan(X_train_t).sum()
nan_test  = np.isnan(X_test_t).sum()
inf_train = np.isinf(X_train_t).sum()
inf_test  = np.isinf(X_test_t).sum()

print(f"\nNaN  — treino: {nan_train}  |  teste: {nan_test}")
print(f"Inf  — treino: {inf_train}  |  teste: {inf_test}")

if nan_train == 0 and nan_test == 0 and inf_train == 0 and inf_test == 0:
    print("\n✅ Nenhum NaN nem Inf no output transformado")

X_train transformado: (679, 100)
X_test  transformado: (170, 100)

NaN  — treino: 0  |  teste: 0
Inf  — treino: 0  |  teste: 0

✅ Nenhum NaN nem Inf no output transformado


---
## Etapa 9 — Reconstrução do DataFrame final

Combina treino e teste num único Parquet com coluna `split` para facilitar
a reprodução dos experimentos pela Pessoa 3 sem necessidade de re-splittar.

Colunas do Parquet final:
- Features transformadas (nomes via `get_feature_names_out`)
- `label` — target binário `label_severe_incident`
- `split` — `"train"` ou `"test"`

In [38]:
# Etapa 9 — Reconstrução do DataFrame final

# Nomes das features após transformação
feature_names = list(preprocessor.get_feature_names_out())
print(f"Total de features pós-pipeline: {len(feature_names)}")

# Inspecionar composição dos nomes
from collections import Counter
prefix_counts = Counter(n.split("__")[0] if "__" in n else "sem_prefixo" for n in feature_names)
print("\nFeatures por transformer (prefixo):")
for prefix, count in prefix_counts.items():
    print(f"  {prefix:30s}: {count}")

print("\nExemplos de nomes:")
for name in feature_names[:5]:
    print(f"  {name}")
print("  ...")
for name in feature_names[-5:]:
    print(f"  {name}")

# Reconstruir DataFrames de treino e teste
df_train = (
    pd.DataFrame(X_train_t, columns=feature_names)
    .assign(label=y_train.values, split="train")
)

df_test = (
    pd.DataFrame(X_test_t, columns=feature_names)
    .assign(label=y_test.values, split="test")
)

# Concatenar com índice contínuo
df_gold = pd.concat([df_train, df_test], ignore_index=True)

print(f"\ndf_gold shape: {df_gold.shape}")
print(f"Colunas 'label' e 'split' presentes: {'label' in df_gold.columns and 'split' in df_gold.columns}")

# Verificação final de integridade
nan_total = df_gold.drop(columns=["split"]).isnull().sum().sum()
print(f"NaN totais no df_gold: {nan_total} {'✅' if nan_total == 0 else '⚠️'}")

dist_split = df_gold["split"].value_counts()
print(f"\nLinhas por split: train={dist_split.get('train', 0)}  |  test={dist_split.get('test', 0)}")

Total de features pós-pipeline: 100

Features por transformer (prefixo):
  sem_prefixo                   : 100

Exemplos de nomes:
  incident_year
  days_to_discovery
  days_to_disclosure
  employee_count
  downtime_hours
  ...
  has_downtime
  is_ransomware
  has_regulatory_fine
  insurance_unknown
  earnings_announcement_within_7d

df_gold shape: (849, 102)
Colunas 'label' e 'split' presentes: True
NaN totais no df_gold: 0 ✅

Linhas por split: train=679  |  test=170


---
## Etapa 10 — Persistência

Salva dois artefatos:
- `data/gold/dataset_ml_ready.parquet` — dataset completo com `label` e `split`
- `models/gold_preprocessor.joblib` — pipeline serializado para uso pela Pessoa 3

In [39]:
# Etapa 10 — Persistência

import joblib

# --- 10a. Parquet ML-ready ---
parquet_path = GOLD_PATH / "dataset_ml_ready.parquet"
df_gold.to_parquet(parquet_path, index=False)
size_mb = parquet_path.stat().st_size / (1024 ** 2)
print(f"✅ dataset_ml_ready.parquet salvo")
print(f"   Caminho : {parquet_path}")
print(f"   Shape   : {df_gold.shape}")
print(f"   Tamanho : {size_mb:.2f} MB")

# --- 10b. Pipeline joblib ---
joblib_path = MODELS_PATH / "gold_preprocessor.joblib"
joblib.dump(preprocessor, joblib_path)
size_kb = joblib_path.stat().st_size / 1024
print(f"\n✅ gold_preprocessor.joblib salvo")
print(f"   Caminho : {joblib_path}")
print(f"   Tamanho : {size_kb:.1f} KB")

✅ dataset_ml_ready.parquet salvo
   Caminho : c:\Users\dti-\Pessoal\CC5\ciencia-de-dados\cybersecurity-breach-data-project\data\gold\dataset_ml_ready.parquet
   Shape   : (849, 102)
   Tamanho : 0.19 MB

✅ gold_preprocessor.joblib salvo
   Caminho : c:\Users\dti-\Pessoal\CC5\ciencia-de-dados\cybersecurity-breach-data-project\models\gold_preprocessor.joblib
   Tamanho : 14.7 KB


---
## Etapa 11 — Validação Final

Verifica todos os critérios de aceitação do plano:

| # | Critério | Verificado aqui |
|---|----------|-----------------|
| 1 | Nenhum NaN no dataset final | ✓ |
| 2 | Schema esperado (features + label + split) | ✓ |
| 3 | Distribuição do label preservada em train/test | ✓ |
| 4 | Pipeline carrega via `joblib.load` e transforma sem erro | ✓ |
| 5 | Output do pipeline recarregado == output original | ✓ |

In [40]:
# Etapa 11 — Validação final

results = {}

# ------------------------------------------------------------------
# 11a. Recarregar o Parquet e verificar ausência de NaN
# ------------------------------------------------------------------
df_check = pd.read_parquet(GOLD_PATH / "dataset_ml_ready.parquet")
nan_total = df_check.drop(columns=["split"]).isnull().sum().sum()
results["sem_nan"] = nan_total == 0
print(f"[{'✅' if results['sem_nan'] else '❌'}] Nenhum NaN no dataset: NaN totais = {nan_total}")

# ------------------------------------------------------------------
# 11b. Schema esperado
# ------------------------------------------------------------------
expected_cols = set(feature_names) | {"label", "split"}
actual_cols   = set(df_check.columns)
missing_cols  = expected_cols - actual_cols
extra_cols    = actual_cols - expected_cols
results["schema_ok"] = (len(missing_cols) == 0 and len(extra_cols) == 0)
print(f"\n[{'✅' if results['schema_ok'] else '❌'}] Schema correto")
if missing_cols:
    print(f"   Colunas ausentes: {missing_cols}")
if extra_cols:
    print(f"   Colunas extras  : {extra_cols}")
print(f"   Total colunas: {len(df_check.columns)} (esperado: {len(expected_cols)})")

# ------------------------------------------------------------------
# 11c. Distribuição do label preservada por split
# ------------------------------------------------------------------
print("\n[📊] Distribuição do label por split:")
label_ok = True
for split_val in ["train", "test"]:
    sub = df_check[df_check["split"] == split_val]["label"]
    dist = sub.value_counts(normalize=True).mul(100).round(1)
    c0, c1 = dist.get(0, 0.0), dist.get(1, 0.0)
    print(f"     {split_val:5s}: classe 0 = {c0}%  |  classe 1 = {c1}%  (n={len(sub)})")
    # Tolerância: proporção original ±2 pp
    if abs(c0 - 13.9) > 2.0:
        label_ok = False
results["label_dist_ok"] = label_ok
print(f"[{'✅' if label_ok else '❌'}] Proporção do label preservada (tolerância ±2 pp)")

# ------------------------------------------------------------------
# 11d. Carregar pipeline via joblib e transformar uma amostra nova
# ------------------------------------------------------------------
loaded_preprocessor = joblib.load(MODELS_PATH / "gold_preprocessor.joblib")

sample = X_test.iloc[:5]
try:
    sample_t = loaded_preprocessor.transform(sample).astype(np.float64)
    shape_ok = sample_t.shape == (5, len(feature_names))
    nan_ok   = np.isnan(sample_t).sum() == 0
    results["joblib_ok"] = shape_ok and nan_ok
    print(f"\n[{'✅' if results['joblib_ok'] else '❌'}] joblib.load + transform: shape={sample_t.shape}, NaN={np.isnan(sample_t).sum()}")
except Exception as e:
    results["joblib_ok"] = False
    print(f"\n[❌] Erro ao usar pipeline recarregado: {e}")

# ------------------------------------------------------------------
# 11e. Output recarregado == output original (5 primeiras linhas do teste)
# ------------------------------------------------------------------
original_sample = X_test_t[:5]
max_diff = np.abs(sample_t - original_sample).max()
results["output_match"] = max_diff < 1e-9
print(f"[{'✅' if results['output_match'] else '❌'}] Output recarregado idêntico ao original (diff máx = {max_diff:.2e})")

# ------------------------------------------------------------------
# Resumo
# ------------------------------------------------------------------
print("\n" + "=" * 50)
print("RESUMO DA VALIDAÇÃO FINAL")
print("=" * 50)
all_pass = all(results.values())
for check, passed in results.items():
    print(f"  {'✅' if passed else '❌'}  {check}")
print("=" * 50)
print(f"  {'✅ TODAS AS VERIFICAÇÕES PASSARAM' if all_pass else '❌ ATENÇÃO: VERIFICAÇÕES FALHARAM'}")

[✅] Nenhum NaN no dataset: NaN totais = 0

[✅] Schema correto
   Total colunas: 102 (esperado: 102)

[📊] Distribuição do label por split:
     train: classe 0 = 13.8%  |  classe 1 = 86.2%  (n=679)
     test : classe 0 = 14.1%  |  classe 1 = 85.9%  (n=170)
[✅] Proporção do label preservada (tolerância ±2 pp)

[✅] joblib.load + transform: shape=(5, 100), NaN=0
[✅] Output recarregado idêntico ao original (diff máx = 0.00e+00)

RESUMO DA VALIDAÇÃO FINAL
  ✅  sem_nan
  ✅  schema_ok
  ✅  label_dist_ok
  ✅  joblib_ok
  ✅  output_match
  ✅ TODAS AS VERIFICAÇÕES PASSARAM


---
## Etapa 12 — Geração de `docs/gold_transformations.md`

Gera programaticamente a tabela de transformações com base nas variáveis do pipeline,
garantindo que a documentação reflita exatamente o que foi implementado.

In [41]:
# Etapa 12 — Geração de docs/gold_transformations.md

import textwrap
from datetime import date

# ------------------------------------------------------------------
# Tabela de transformações por grupo — uma entrada por coluna
# ------------------------------------------------------------------
# Estrutura: (coluna, grupo, etapas_pipeline, técnica, justificativa)
TRANSFORM_TABLE = []

# ── num_normal: IQRClipper → SimpleImputer(median, add_indicator) → StandardScaler
_num_normal_meta = {
    "incident_year":             "Escala temporal uniforme; sem outliers extremos esperados",
    "days_to_discovery":         "Outliers altos (incidentes não detectados por anos); IQR clip necessário",
    "days_to_disclosure":        "Idem; janela legal/regulatória pode criar caudas longas",
    "employee_count":            "Outliers de mega-corporações distorcem splits de DT",
    "downtime_hours":            "Valores extremos pontuais; IQR clip reduz distorção no DT",
    "data_compromised_records":  "Distribuição assimétrica; clipping antes da imputação",
    "volume_avg_30d_baseline":   "Volume de negociação com outliers de dias anômalos",
    "volume_disclosure_day":     "Idem; pico no dia do disclosure pode ser extremo",
    "sector_return_same_period": "Retorno setorial; outliers em crises de mercado",
    "t_statistic_1d":            "Estatística t pode ter valores extremos em amostras pequenas",
    "p_value_1d":                "Valores em [0,1]; sem outliers severos — clip inerte mas consistente",
    "t_statistic_30d":           "Idem t_statistic_1d para janela de 30d",
    "p_value_30d":               "Idem p_value_1d para janela de 30d",
    "volume_ratio_disclosure":   "Ratio pode ter valores extremos em pânico de mercado",
    "pre_incident_volatility_30d": "Volatilidade; outliers em períodos de alta turbulência",
}
for col, justif in _num_normal_meta.items():
    TRANSFORM_TABLE.append({
        "coluna": f"`{col}`",
        "grupo":  "num_normal",
        "etapas": "Outliers → Missing → Scaling",
        "tecnica": "IQRClipper(k=1.5) → SimpleImputer(median, add_indicator=True) → StandardScaler",
        "justificativa": justif,
        "fit_em": "treino",
    })

# ── num_monetary: SimpleImputer(median, add_indicator) → log1p → RobustScaler
_num_monetary_meta = {
    "company_revenue_usd":   "Escala de bilhões; log1p comprime; NaN estrutural (empresa não reportou)",
    "direct_loss_usd":       "Cauda longa documentada na EDA (G5); 8,5% sem dados financeiros",
    "ransom_demanded_usd":   "73,5% nulos (só ransomware); mediana + flag; cauda extrema",
    "ransom_paid_usd":       "88,9% nulos; idem ransom_demanded",
    "recovery_cost_usd":     "Distribuição assimétrica; NaN para incidentes sem custo documentado",
    "legal_fees_usd":        "Idem recovery_cost; alta variação por jurisdição",
    "regulatory_fine_usd":   "83,0% nulos; só incidentes com multa regulatória",
    "insurance_payout_usd":  "NaN quando não há apólice; mediana + flag semânticamente correto",
    "total_loss_usd":        "Feature principal de impacto financeiro (EDA G5); log1p obrigatório",
    "total_loss_lower_bound":"Correlacionada com total_loss; idem tratamento",
    "total_loss_upper_bound":"Idem; cauda superior longa",
    "inflation_adjusted_usd":"Idem total_loss após ajuste; log1p correto",
    "price_7d_before":       "Preço pré-evento em USD; escala monetária; NaN para não-públicas",
    "price_disclosure_day":  "Idem; disponível apenas para empresas públicas (58% nulos)",
    "market_cap_at_disclosure": "Capitalização em USD bilhões; log1p essencial; NaN para não-públicas",
}
for col, justif in _num_monetary_meta.items():
    TRANSFORM_TABLE.append({
        "coluna": f"`{col}`",
        "grupo":  "num_monetary",
        "etapas": "Missing → Outliers/Compressão → Scaling",
        "tecnica": "SimpleImputer(median, add_indicator=True) → log1p (FunctionTransformer) → RobustScaler",
        "justificativa": justif,
        "fit_em": "treino",
    })

# ── cat_lowcard: SimpleImputer(constant='unknown') → OneHotEncoder(min_frequency=10)
_cat_lowcard_meta = {
    "attack_vector_primary":  "Baixa cardinalidade; categorias nominais sem ordem; min_freq=10 agrupa raros",
    "attribution_confidence": "3–4 níveis ordinais tratados como nominais para simplicidade",
    "data_type":              "Tipo de dado comprometido; nominal; sem ordem natural",
    "data_source_type":       "Fonte de coleta do incidente; categórica nominal",
    "sector_index":           "10 setores S&P; baixa cardinalidade; OHE apropriado",
}
for col, justif in _cat_lowcard_meta.items():
    TRANSFORM_TABLE.append({
        "coluna": f"`{col}`",
        "grupo":  "cat_lowcard",
        "etapas": "Missing → Encoding",
        "tecnica": "SimpleImputer(constant='unknown') → OneHotEncoder(handle_unknown='ignore', min_frequency=10)",
        "justificativa": justif,
        "fit_em": "treino",
    })

# ── cat_highcard: SimpleImputer(constant='unknown') → TargetEncoder
_cat_highcard_meta = {
    "industry_primary": "Alta cardinalidade (~20 setores industriais); TargetEncoder evita explosão dimensional",
    "country_hq":       "50+ países; OHE criaria >50 colunas esparsas; TargetEncoder mantém 1 coluna",
    "attributed_group": "Grupos de ameaça variados; muitos valores raros; encoding supervisionado adequado",
}
for col, justif in _cat_highcard_meta.items():
    TRANSFORM_TABLE.append({
        "coluna": f"`{col}`",
        "grupo":  "cat_highcard",
        "etapas": "Missing → Encoding",
        "tecnica": "SimpleImputer(constant='unknown') → TargetEncoder(target_type='binary')",
        "justificativa": justif,
        "fit_em": "treino",
    })

# ── binary_cols: SimpleImputer(constant=0)
_binary_meta = {
    "is_public_company":             "Flag 0/1; NaN ausente nos dados; passthrough após imputação",
    "incident_date_estimated":       "Indica qualidade da data; bool; NaN estrutural → 0",
    "has_secondary_vector":          "Flag de complexidade do ataque; 0/1",
    "data_loss_unknown":             "Flag de incerteza; 0/1; criada na Silver",
    "downtime_unknown":              "Idem; indica se downtime foi reportado",
    "has_data_loss":                 "Target auxiliar de impacto; 0/1",
    "has_downtime":                  "Idem para downtime",
    "is_ransomware":                 "Flag de tipo de ataque; NaN → 0 (sem dados financeiros = sem ransomware)",
    "has_regulatory_fine":           "Flag de multa; NaN → 0 (sem dados financeiros = sem multa registrada)",
    "insurance_unknown":             "Flag de incerteza do seguro; NaN → 0",
    "earnings_announcement_within_7d": "Flag de confundidor; bool; NaN para não-públicas → 0",
}
for col, justif in _binary_meta.items():
    TRANSFORM_TABLE.append({
        "coluna": f"`{col}`",
        "grupo":  "binary_cols",
        "etapas": "Missing",
        "tecnica": "SimpleImputer(constant=0)",
        "justificativa": justif,
        "fit_em": "treino",
    })

# ------------------------------------------------------------------
# Seção de leakage removido
# ------------------------------------------------------------------
LEAKAGE_TABLE = [
    ("incident_id",                 "todas",    "Identificador — não é feature"),
    ("stock_ticker",                "incidents","Identificador de empresa"),
    ("stock_ticker_mkt",            "market",   "Duplicata de stock_ticker após join"),
    ("incident_date",               "incidents","Data crua — derivada em days_to_*"),
    ("discovery_date",              "incidents","Data crua — derivada em days_to_discovery"),
    ("incident_month",              "incidents","Granularidade sem valor preditivo para DT"),
    ("incident_day",                "incidents","Idem incident_month"),
    ("price_1d_after",              "market",   "Preço pós-evento → leakage em produção"),
    ("price_7d_after",              "market",   "Idem"),
    ("price_30d_after",             "market",   "Idem"),
    ("abnormal_return_1d",          "market",   "Calculado com preço pós-evento → leakage"),
    ("abnormal_return_7d",          "market",   "Idem"),
    ("abnormal_return_30d",         "market",   "Idem"),
    ("car_neg1_to_pos1",            "market",   "CAR inclui pós-evento → leakage"),
    ("car_0_to_7",                  "market",   "Idem"),
    ("car_0_to_30",                 "market",   "Idem"),
    ("car_0_to_90",                 "market",   "Idem"),
    ("post_incident_volatility_30d","market",   "Calculado após incidente → leakage"),
    ("days_to_price_recovery",      "market",   "Só disponível retroativamente → leakage"),
]

# ------------------------------------------------------------------
# Gerar markdown
# ------------------------------------------------------------------
def md_row(cells):
    return "| " + " | ".join(str(c) for c in cells) + " |"

lines = [
    "# Gold Transformations",
    "",
    f"> Gerado automaticamente pelo notebook `gold_pipeline.ipynb` em {date.today().isoformat()}.",
    "",
    "---",
    "",
    "## Pipeline de transformações por coluna",
    "",
    "| Coluna | Grupo | Etapas | Técnica | Justificativa | Fit em |",
    "|--------|-------|--------|---------|---------------|--------|",
]
for row in TRANSFORM_TABLE:
    lines.append(md_row([
        row["coluna"], row["grupo"], row["etapas"],
        row["tecnica"], row["justificativa"], row["fit_em"],
    ]))

lines += [
    "",
    "---",
    "",
    "## Colunas removidas por leakage (Gold)",
    "",
    "| Coluna | Dataset de origem | Motivo |",
    "|--------|-------------------|--------|",
]
for col, origem, motivo in LEAKAGE_TABLE:
    lines.append(md_row([f"`{col}`", origem, motivo]))

lines += [
    "",
    "---",
    "",
    "## Artefatos gerados",
    "",
    "| Artefato | Caminho | Descrição |",
    "|----------|---------|-----------|",
    "| Dataset ML-ready | `data/gold/dataset_ml_ready.parquet` | 849 × 102 (100 features + label + split) |",
    "| Parquet retroativo | `data/gold/market_retroactive.parquet` | 357 × 13 (colunas pós-evento para análise futura) |",
    "| Pipeline serializado | `models/gold_preprocessor.joblib` | ColumnTransformer fitado no treino |",
    "",
    "---",
    "",
    "## Split",
    "",
    "| Conjunto | Linhas | Proporção classe 0 | Proporção classe 1 |",
    "|----------|--------|--------------------|--------------------|",
    "| treino   | 679    | 13.8%              | 86.2%              |",
    "| teste    | 170    | 14.1%              | 85.9%              |",
    "",
    "> Split estratificado com `random_state=42`. Fit realizado exclusivamente sobre o treino.",
]

doc_path = DOCS_PATH / "gold_transformations.md"
doc_path.write_text("\n".join(lines), encoding="utf-8")

print(f"✅ docs/gold_transformations.md gerado")
print(f"   Linhas : {len(lines)}")
print(f"   Colunas documentadas: {len(TRANSFORM_TABLE)}")
print(f"   Colunas de leakage  : {len(LEAKAGE_TABLE)}")

✅ docs/gold_transformations.md gerado
   Linhas : 106
   Colunas documentadas: 49
   Colunas de leakage  : 19
